<a href="https://colab.research.google.com/github/SarahkhIT/AgenticAIProject/blob/main/AgenticAIProject/notebooks/01_agent_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Event Planner — 01. Agent Fundamentals

**Program:** Building Agentic AI Systems by SDAIA Academy

**Session Dates:** 9th of August, 2026 - 13th of August, 2026

**Declared Track:** Track A

**Covers:** Rubric 1 — Agent Fundamentals

A real LangChain agent with model-selected tools (budget, venue, catering, decoration, checklist) plus structured intake using a Pydantic `EventRequest` model.

## Team Members
- Setah Mohammed Alajmi
- Raneem Abdullah Alsheddi
- Jana Hamad Alhumaizi
- Shatha Hamad Bin Mana
- Sarah Abdulaziz Alkhudhiri


In [ ]:
# ============================================================
# SUBMISSION IDENTITY
# ============================================================

TEAM_MEMBERS = [
"Shatha Hamad Bin Mana" ,
"Setah Mohammed Alajmi",
"Raneem Abdullah Alsheddi",
"Jana Hamad Alhumaizi",
"Sarah Abdulaziz Alkhudhiri",
                 ]

print("Declared Track: Track A")
print("Team members entered:", len(TEAM_MEMBERS))


Declared Track: Track A
Team members entered: 5


In [ ]:
!pip install -q \
    "langchain>=1.0" \
    "langgraph>=1.0" \
    langchain-groq \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    pypdf \
    reportlab

In [ ]:
# ============================================================
# CAPSTONE SECRET SETUP — NO HARDCODED API KEYS
# ============================================================

import os
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")

if not groq_key:
    raise RuntimeError(
        "GROQ_API_KEY is missing. Add it to Colab Secrets and enable Notebook access."
    )

os.environ["GROQ_API_KEY"] = groq_key

print("PASS: GROQ_API_KEY loaded securely from Colab Secrets.")


PASS: GROQ_API_KEY loaded securely from Colab Secrets.


# Rubric 1 — Agent Fundamentals

**Pattern used:** a real LangChain agent with model-selected tools plus structured intake.

The Smart Event Planner exposes five real event-planning tools whose arguments affect the returned budget, venue, catering, decoration, and checklist results. Natural-language event requests are parsed into the `EventRequest` Pydantic model with `with_structured_output()`, so code consumes validated structured data rather than parsing free-form text. The capstone demo below is the evidence cell: it must show the LLM selecting and calling the tools, the arguments/results, the structured `EventRequest`, and the final response.


In [ ]:
# ============================================================
# SMART EVENT PLANNER
# Person 1 — Agent Fundamentals & Tools
# ============================================================
#
# Capstone requirements addressed in this file:
#
# 1. Agent Fundamentals
#    - Real LLM-based agent
#    - Real tool calls
#    - Tools use their arguments to perform actual work
#    - Structured output with Pydantic
#    - with_structured_output()
#
# 2. Integration-ready
#    - EventRequest is a clear contract
#    - build_event_planning_agent() can be used by Person 2
#    - Tools are independent and reusable
#
# IMPORTANT:
# - Set GROQ_API_KEY as an environment variable.
# - Never put the API key directly in this file.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
from typing import Literal

from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# ============================================================
# 1. CONFIGURATION
# ============================================================

DEFAULT_MODEL = "groq:llama-3.3-70b-versatile"


def get_model(model_name: str = DEFAULT_MODEL):
    """
    Create the chat model.

    The API key is read from the environment.
    """

    if not os.environ.get("GROQ_API_KEY"):
        raise RuntimeError(
            "GROQ_API_KEY is not set.\n"
            "Please set it as an environment variable before running the agent."
        )

    return init_chat_model(model_name)


# ============================================================
# 2. PYDANTIC MODELS
# ============================================================

class EventRequest(BaseModel):
    """
    Structured representation of the user's event request.

    The LLM populates this model using with_structured_output().
    """

    event_type: str = Field(
        description="Type of event, for example graduation, wedding, birthday."
    )

    guest_count: int = Field(
        gt=0,
        description="Expected number of guests."
    )

    budget: float = Field(
        gt=0,
        description="Total event budget in Saudi Riyals."
    )

    date: str | None = Field(
        default=None,
        description="Event date if explicitly provided by the user."
    )

    location_pref: Literal["indoor", "outdoor"] | None = Field(
        default=None,
        description="Indoor or outdoor preference if explicitly provided."
    )

    theme: str | None = Field(
        default=None,
        description="Event theme or style if explicitly provided."
    )


class BudgetBreakdown(BaseModel):
    venue: float
    catering: float
    decoration: float
    entertainment: float
    logistics: float
    contingency: float


class VenueOption(BaseModel):
    name: str
    capacity: int
    indoor: bool
    estimated_cost: float
    location: str
    reason: str


class CateringOption(BaseModel):
    name: str
    price_per_guest: float
    estimated_total: float
    cuisine: str
    reason: str


class DecorationPlan(BaseModel):
    theme: str
    estimated_cost: float
    concept: str
    items: list[str]


class ChecklistItem(BaseModel):
    task: str
    category: str
    due_before_event_days: int
    priority: Literal["high", "medium", "low"]


# ============================================================
# 3. TOOL 1 — BUDGET CALCULATOR
# ============================================================

@tool
def calculate_budget_split(
    total_budget: float,
    event_type: str,
) -> dict:
    """
    Calculate a realistic event budget allocation.

    Arguments:
        total_budget: Total event budget in SAR.
        event_type: Type of event.

    Returns:
        A structured budget breakdown.
    """

    if total_budget <= 0:
        raise ValueError("total_budget must be greater than zero.")

    if not event_type.strip():
        raise ValueError("event_type cannot be empty.")

    # Default allocation for the Smart Event Planner.
    #
    # The event_type is intentionally accepted as an argument
    # because the LLM must provide it when calling the tool.
    #
    # This can later be customized for different event types.

    allocation = {
        "venue": 0.30,
        "catering": 0.35,
        "decoration": 0.15,
        "entertainment": 0.08,
        "logistics": 0.07,
        "contingency": 0.05,
    }

    breakdown = BudgetBreakdown(
        venue=round(total_budget * allocation["venue"], 2),
        catering=round(total_budget * allocation["catering"], 2),
        decoration=round(total_budget * allocation["decoration"], 2),
        entertainment=round(total_budget * allocation["entertainment"], 2),
        logistics=round(total_budget * allocation["logistics"], 2),
        contingency=round(total_budget * allocation["contingency"], 2),
    )

    return breakdown.model_dump()


# ============================================================
# 4. TOOL 2 — VENUE SEARCH
# ============================================================

@tool
def search_venues(
    guest_count: int,
    budget: float,
    indoor: bool,
) -> list[dict]:
    """
    Find venues based on guest capacity, budget and indoor/outdoor preference.

    Arguments:
        guest_count: Number of guests.
        budget: Maximum venue budget in SAR.
        indoor: True for indoor venues, False for outdoor venues.

    Returns:
        Matching venue options.
    """

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    # Demo dataset.
    #
    # These are NOT claimed to be real external venues.
    # They are project data used to demonstrate real filtering
    # and tool execution.
    venues = [
        {
            "name": "Elegant Hall Riyadh",
            "capacity": 120,
            "indoor": True,
            "estimated_cost": 4200,
            "location": "Riyadh",
            "reason": "Elegant indoor hall suitable for formal celebrations.",
        },
        {
            "name": "Garden Celebration Venue",
            "capacity": 150,
            "indoor": False,
            "estimated_cost": 3500,
            "location": "Riyadh",
            "reason": "Outdoor garden venue suitable for large celebrations.",
        },
        {
            "name": "Modern Event Studio",
            "capacity": 90,
            "indoor": True,
            "estimated_cost": 3900,
            "location": "Riyadh",
            "reason": "Modern indoor venue suitable for smaller elegant events.",
        },
        {
            "name": "Grand Celebration Center",
            "capacity": 250,
            "indoor": True,
            "estimated_cost": 5500,
            "location": "Riyadh",
            "reason": "Large indoor event center for bigger celebrations.",
        },
    ]

    # REAL filtering using the tool arguments.
    matching_venues = [
        venue
        for venue in venues
        if (
            venue["capacity"] >= guest_count
            and venue["estimated_cost"] <= budget
            and venue["indoor"] == indoor
        )
    ]

    # Cheapest suitable options first.
    matching_venues.sort(
        key=lambda venue: (
            venue["estimated_cost"],
            venue["capacity"],
        )
    )

    return [
        VenueOption(**venue).model_dump()
        for venue in matching_venues[:3]
    ]


# ============================================================
# 5. TOOL 3 — CATERING SEARCH
# ============================================================

@tool
def search_catering(
    guest_count: int,
    budget: float,
) -> list[dict]:
    """
    Find catering options based on guest count and catering budget.

    Arguments:
        guest_count: Number of guests.
        budget: Maximum catering budget in SAR.

    Returns:
        Catering options that fit the budget.
    """

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    catering_options = [
        {
            "name": "Premium Saudi Buffet",
            "price_per_guest": 65,
            "cuisine": "Saudi / Arabic",
            "reason": "Suitable for formal family celebrations.",
        },
        {
            "name": "International Buffet",
            "price_per_guest": 70,
            "cuisine": "International",
            "reason": "Broad menu suitable for mixed preferences.",
        },
        {
            "name": "Elegant Finger Food",
            "price_per_guest": 45,
            "cuisine": "International",
            "reason": "Suitable for a modern elegant reception.",
        },
    ]

    results = []

    for option in catering_options:

        # REAL calculation based on guest_count.
        estimated_total = (
            guest_count * option["price_per_guest"]
        )

        # REAL budget filtering.
        if estimated_total <= budget:

            results.append(
                CateringOption(
                    name=option["name"],
                    price_per_guest=option["price_per_guest"],
                    estimated_total=estimated_total,
                    cuisine=option["cuisine"],
                    reason=option["reason"],
                ).model_dump()
            )

    results.sort(
        key=lambda item: item["estimated_total"]
    )

    return results[:3]


# ============================================================
# 6. TOOL 4 — DECORATION PLANNER
# ============================================================

@tool
def suggest_decoration(
    theme: str,
    guest_count: int,
    budget: float,
) -> dict:
    """
    Suggest a decoration concept based on theme, guest count and budget.

    Arguments:
        theme: Requested event theme.
        guest_count: Number of guests.
        budget: Decoration budget in SAR.

    Returns:
        A structured decoration plan.
    """

    if not theme.strip():
        raise ValueError("theme cannot be empty.")

    if guest_count <= 0:
        raise ValueError("guest_count must be greater than zero.")

    if budget <= 0:
        raise ValueError("budget must be greater than zero.")

    normalized_theme = theme.lower().strip()

    if "elegant" in normalized_theme:

        concept = (
            "Elegant graduation setup with neutral colors, "
            "warm lighting, floral accents and a decorated stage."
        )

        items = [
            "Graduation backdrop",
            "Warm ambient lighting",
            "Floral centerpieces",
            "Welcome signage",
            "Decorated stage",
        ]

    elif "modern" in normalized_theme:

        concept = (
            "Modern minimalist setup with clean lines, "
            "accent lighting and a contemporary photo area."
        )

        items = [
            "Minimalist backdrop",
            "Accent lighting",
            "Modern table styling",
            "Photo wall",
        ]

    elif "casual" in normalized_theme:

        concept = (
            "Casual and welcoming setup with simple colors, "
            "comfortable seating and themed decorations."
        )

        items = [
            "Themed backdrop",
            "Simple table decoration",
            "Welcome signage",
            "Photo area",
        ]

    else:

        concept = (
            f"A {theme} themed decoration concept "
            "adapted to the event size and available budget."
        )

        items = [
            "Themed backdrop",
            "Table decoration",
            "Welcome signage",
            "Photo area",
        ]

    return DecorationPlan(
        theme=theme,
        estimated_cost=round(budget, 2),
        concept=concept,
        items=items,
    ).model_dump()


# ============================================================
# 7. TOOL 5 — EVENT CHECKLIST
# ============================================================

@tool
def create_checklist(
    event_date: str,
    categories: list[str],
) -> list[dict]:
    """
    Create an event preparation checklist.

    Arguments:
        event_date: Event date.
        categories: Planning categories to include.

    Returns:
        Checklist items.
    """

    if not event_date.strip():
        raise ValueError("event_date is required.")

    if not categories:
        raise ValueError("At least one category is required.")

    category_tasks = {

        "venue": [
            (
                "Confirm venue booking",
                30,
                "high",
            ),
            (
                "Confirm seating layout",
                7,
                "medium",
            ),
        ],

        "catering": [
            (
                "Confirm catering menu",
                14,
                "high",
            ),
            (
                "Confirm final guest count",
                3,
                "high",
            ),
        ],

        "decoration": [
            (
                "Finalize decoration concept",
                21,
                "medium",
            ),
            (
                "Confirm decoration setup",
                7,
                "high",
            ),
        ],

        "logistics": [
            (
                "Prepare event timeline",
                7,
                "high",
            ),
            (
                "Confirm equipment and sound system",
                5,
                "medium",
            ),
        ],

        "approval": [
            (
                "Review final event plan with user",
                1,
                "high",
            ),
        ],
    }

    checklist = []

    for category in categories:

        tasks = category_tasks.get(
            category.lower().strip(),
            [],
        )

        for task, days, priority in tasks:

            checklist.append(
                ChecklistItem(
                    task=task,
                    category=category,
                    due_before_event_days=days,
                    priority=priority,
                ).model_dump()
            )

    return checklist


# ============================================================
# 8. REGISTER ALL TOOLS
# ============================================================

ALL_TOOLS = [
    calculate_budget_split,
    search_venues,
    search_catering,
    suggest_decoration,
    create_checklist,
]


# ============================================================
# 9. STRUCTURED INTAKE
# ============================================================

INTAKE_SYSTEM_PROMPT = """
You are the structured intake component of Smart Event Planner.

Extract event-planning information from the user's message.

IMPORTANT:
Only extract information that is explicitly supported by the user's
message.

DO NOT invent or assume:
- event dates
- guest counts
- budgets
- indoor/outdoor preferences
- themes

The required event fields are:
- event_type
- guest_count
- budget

The optional fields are:
- date
- location_pref
- theme

If an optional field is not mentioned, return null.

The budget is represented in Saudi Riyals.
"""


def extract_event_request(
    user_message: str,
    model=None,
) -> EventRequest:
    """
    Convert natural-language user input into a validated EventRequest.

    This uses LangChain structured output with a Pydantic model.
    """

    if not user_message.strip():
        raise ValueError(
            "user_message cannot be empty."
        )

    model = model or get_model()

    structured_model = model.with_structured_output(
        EventRequest
    )

    event = structured_model.invoke(
        [
            {
                "role": "system",
                "content": INTAKE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ]
    )

    return event


# ============================================================
# 10. CHECK WHETHER EVENT DETAILS ARE COMPLETE
# ============================================================

def get_missing_event_fields(
    event: EventRequest,
) -> list[str]:
    """
    Return optional fields that are useful for complete planning.

    This does NOT invent missing information.
    """

    missing = []

    if event.date is None:
        missing.append("date")

    if event.location_pref is None:
        missing.append("location_pref")

    if event.theme is None:
        missing.append("theme")

    return missing


def build_clarifying_question(
    event: EventRequest,
) -> str | None:
    """
    Build a follow-up question when optional planning details
    are missing.
    """

    missing = get_missing_event_fields(event)

    if not missing:
        return None

    questions = []

    if "date" in missing:
        questions.append(
            "What date is the event?"
        )

    if "location_pref" in missing:
        questions.append(
            "Would you prefer indoor or outdoor?"
        )

    if "theme" in missing:
        questions.append(
            "What style or theme would you like?"
        )

    return (
        "Before I build the full plan, "
        "please provide: "
        + " ".join(questions)
    )


# ============================================================
# 11. AGENT SYSTEM PROMPT
# ============================================================

AGENT_SYSTEM_PROMPT = """
You are the core event-planning agent for Smart Event Planner.

Your job is to create a practical event plan using the available tools.

IMPORTANT RULE:
Use the tools to obtain planning information.
Do NOT invent:
- budget calculations
- venue options
- catering prices
- decoration costs
- checklist items

The available tools are:

1. calculate_budget_split
   Use it to calculate the event budget allocation.

2. search_venues
   Use it to find suitable venues based on guest count,
   venue budget and indoor/outdoor preference.

3. search_catering
   Use it to find catering options based on guest count
   and catering budget.

4. suggest_decoration
   Use it to create a decoration concept based on theme,
   guest count and decoration budget.

5. create_checklist
   Use it to create event preparation tasks.

You should call the tools yourself when needed.

The tool results are the source of truth for numerical
recommendations.

When all relevant information is available, produce a concise
event plan containing:

1. Event summary
2. Budget breakdown
3. Recommended venue
4. Catering recommendation
5. Decoration concept
6. Preparation checklist

Do not claim that the venue or catering data came from a real
external service. The current tools use the project's local
planning dataset.

If a required detail is missing, clearly state that the information
is needed rather than inventing it.
"""


# ============================================================
# 12. BUILD THE REAL AGENT
# ============================================================

def build_event_planning_agent(
    model=None,
):
    """
    Build the reusable Smart Event Planner agent.

    Person 2 can later integrate this agent into the project's
    multi-agent routing architecture.
    """

    model = model or get_model()

    agent = create_agent(
        model=model,
        tools=ALL_TOOLS,
        system_prompt=AGENT_SYSTEM_PROMPT,
    )

    return agent


# ============================================================
# 13. RUN THE AGENT
# ============================================================

def run_event_planner(
    event: EventRequest,
    model=None,
):
    """
    Run the event planning agent using a validated EventRequest.
    """

    # The planning agent needs these details.
    if event.date is None:
        raise ValueError(
            "event.date is required before running the full planner."
        )

    if event.location_pref is None:
        raise ValueError(
            "event.location_pref is required before running the full planner."
        )

    if event.theme is None:
        raise ValueError(
            "event.theme is required before running the full planner."
        )

    agent = build_event_planning_agent(model)

    event_details = event.model_dump()

    planning_prompt = f"""
Create the event plan using the following structured event request:

{event_details}

You must use the available tools to calculate and retrieve
the planning information.

Do not invent tool results.

Use the actual tool outputs in your final response.
"""

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": planning_prompt,
                }
            ]
        }
    )

    return result


# ============================================================
# 14. DISPLAY REAL TOOL CALLS
# ============================================================

def display_agent_trace(
    result,
):
    """
    Display the actual LLM -> Tool -> Tool Result execution.

    This is important evidence for the Capstone rubric because
    it demonstrates that the LLM selected and called the tools.
    """

    print("\n")
    print("=" * 80)
    print("REAL AGENT TOOL-CALL TRACE")
    print("=" * 80)

    tool_call_count = 0

    for message in result["messages"]:

        # ----------------------------------------------------
        # AI MESSAGE
        # ----------------------------------------------------

        if message.type == "ai":

            tool_calls = getattr(
                message,
                "tool_calls",
                [],
            )

            if tool_calls:

                for call in tool_calls:

                    tool_call_count += 1

                    print("\n[LLM -> TOOL]")
                    print(
                        f"Tool: {call['name']}"
                    )

                    print(
                        "Arguments:"
                    )

                    print(
                        call["args"]
                    )

            elif message.content:

                print("\n[LLM FINAL RESPONSE]")
                print(message.content)

        # ----------------------------------------------------
        # TOOL MESSAGE
        # ----------------------------------------------------

        elif message.type == "tool":

            print("\n[TOOL -> LLM]")

            print(
                f"Tool: {message.name}"
            )

            print(
                "Result:"
            )

            print(
                message.content
            )

    print("\n")
    print("=" * 80)
    print(
        f"TOTAL TOOL CALLS: {tool_call_count}"
    )
    print("=" * 80)

    return tool_call_count


# ============================================================
# 15. OFFLINE TOOL TEST
# ============================================================
#
# This section verifies that the tools themselves work.
#
# IMPORTANT:
# These are MANUAL tool calls.
# They are NOT the evidence for LLM tool selection.
#
# The actual Agent evidence is the test below this section.
# ============================================================

def run_offline_tool_tests():

    print("\n")
    print("=" * 80)
    print("OFFLINE TOOL TESTS")
    print("=" * 80)

    budget = calculate_budget_split.invoke(
        {
            "total_budget": 15000,
            "event_type": "graduation",
        }
    )

    print("\nBudget:")
    print(budget)

    venues = search_venues.invoke(
        {
            "guest_count": 80,
            "budget": budget["venue"],
            "indoor": True,
        }
    )

    print("\nVenues:")
    for venue in venues:
        print(venue)

    catering = search_catering.invoke(
        {
            "guest_count": 80,
            "budget": budget["catering"],
        }
    )

    print("\nCatering:")
    for option in catering:
        print(option)

    decoration = suggest_decoration.invoke(
        {
            "theme": "elegant",
            "guest_count": 80,
            "budget": budget["decoration"],
        }
    )

    print("\nDecoration:")
    print(decoration)

    checklist = create_checklist.invoke(
        {
            "event_date": "2026-09-20",
            "categories": [
                "venue",
                "catering",
                "decoration",
                "logistics",
                "approval",
            ],
        }
    )

    print("\nChecklist:")
    for item in checklist:
        print(item)

    print("\nOffline tool tests completed successfully.")


# ============================================================
# 16. CAPSTONE DEMO
# ============================================================

def run_capstone_demo():

    print("\n")
    print("#" * 80)
    print("# SMART EVENT PLANNER — PERSON 1 CAPSTONE DEMO")
    print("#" * 80)

    # --------------------------------------------------------
    # USER INPUT
    # --------------------------------------------------------

    user_message = """
    I want to plan a graduation party for 80 guests
    with a budget of 15,000 SAR.
    The event will be on 20 September 2026.
    I prefer an indoor venue with an elegant theme.
    """

    print("\n")
    print("=" * 80)
    print("USER REQUEST")
    print("=" * 80)

    print(user_message)

    # --------------------------------------------------------
    # STRUCTURED INTAKE
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("STRUCTURED OUTPUT — EVENT REQUEST")
    print("=" * 80)

    event = extract_event_request(
        user_message
    )

    print(
        event.model_dump_json(
            indent=2
        )
    )

    # --------------------------------------------------------
    # CHECK MISSING INFORMATION
    # --------------------------------------------------------

    missing = get_missing_event_fields(
        event
    )

    if missing:

        print("\n")
        print("=" * 80)
        print("MISSING OPTIONAL INFORMATION")
        print("=" * 80)

        print(
            build_clarifying_question(event)
        )

        print(
            "\nThe demo will stop here because the planner "
            "does not invent missing information."
        )

        return

    # --------------------------------------------------------
    # REAL AGENT EXECUTION
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("RUNNING REAL LLM AGENT")
    print("=" * 80)

    result = run_event_planner(
        event
    )

    # --------------------------------------------------------
    # DISPLAY REAL TOOL CALLS
    # --------------------------------------------------------

    tool_call_count = display_agent_trace(
        result
    )

    # --------------------------------------------------------
    # CAPSTONE ASSERTION
    # --------------------------------------------------------

    if tool_call_count == 0:

        raise RuntimeError(
            "The agent completed without calling any tool. "
            "This would NOT satisfy the real tool-calling requirement."
        )

    print("\n")
    print("=" * 80)
    print("CAPSTONE CHECK")
    print("=" * 80)

    print(
        "PASS: The LLM made real tool calls."
    )

    print(
        f"PASS: Number of tool calls = {tool_call_count}"
    )

    print(
        "PASS: Structured EventRequest was produced with Pydantic."
    )

    print(
        "PASS: with_structured_output() was used."
    )

    print(
        "PASS: Tools received arguments from the agent."
    )

    print(
        "PASS: Tool results were returned to the LLM."
    )

    print(
        "PASS: Final plan was generated from tool results."
    )


# ============================================================
# 17. MAIN
# ============================================================

if __name__ == "__main__":

    print(
        "Smart Event Planner — Person 1")

    print( "Agent Fundamentals & Tools")

    print( "\nAvailable components:")

    print( "- EventRequest")

    print("- Structured Intake")

    print("- calculate_budget_split")

    print("- search_venues")

    print("- search_catering")

    print( "- suggest_decoration")

    print( "- create_checklist")

    print("- Event Planning Agent")

    print("\nRun run_offline_tool_tests() to test tools." )

    print( "Run run_capstone_demo() to demonstrate the real agent.")

Smart Event Planner — Person 1
Agent Fundamentals & Tools

Available components:
- EventRequest
- Structured Intake
- calculate_budget_split
- search_venues
- search_catering
- suggest_decoration
- create_checklist
- Event Planning Agent

Run run_offline_tool_tests() to test tools.
Run run_capstone_demo() to demonstrate the real agent.


In [ ]:
# ============================================================
# PERSON 1 — GROQ-COMPATIBLE TOOL WRAPPERS
# ============================================================

import json


@tool
def demo_budget(total_budget: float, event_type: str) -> str:
    """Calculate the event budget allocation."""
    result = calculate_budget_split.invoke({
        "total_budget": total_budget,
        "event_type": event_type,
    })
    return json.dumps(result, ensure_ascii=False)


@tool
def demo_venues(
    guest_count: int,
    budget: float,
    indoor: bool,
) -> str:
    """Find suitable venues."""
    result = search_venues.invoke({
        "guest_count": guest_count,
        "budget": budget,
        "indoor": indoor,
    })
    return json.dumps(result, ensure_ascii=False)


@tool
def demo_catering(
    guest_count: int,
    budget: float,
) -> str:
    """Find suitable catering options."""
    result = search_catering.invoke({
        "guest_count": guest_count,
        "budget": budget,
    })
    return json.dumps(result, ensure_ascii=False)


@tool
def demo_decoration(
    theme: str,
    guest_count: int,
    budget: float,
) -> str:
    """Create a decoration concept."""
    result = suggest_decoration.invoke({
        "theme": theme,
        "guest_count": guest_count,
        "budget": budget,
    })
    return json.dumps(result, ensure_ascii=False)


@tool
def demo_checklist(
    event_date: str,
    categories: list[str],
) -> str:
    """Create an event preparation checklist."""
    result = create_checklist.invoke({
        "event_date": event_date,
        "categories": categories,
    })
    return json.dumps(result, ensure_ascii=False)


PERSON1_GROQ_TOOLS = [
    demo_budget,
    demo_venues,
    demo_catering,
    demo_decoration,
    demo_checklist,
]

print("Person 1 Groq-compatible tools created successfully.")

Person 1 Groq-compatible tools created successfully.


In [ ]:
# ============================================================
# REBUILD PERSON 1 AGENT FOR GROQ
# ============================================================

def build_event_planning_agent(model=None):
    model = model or get_model()

    agent = create_agent(
        model=model,
        tools=PERSON1_GROQ_TOOLS,
        system_prompt=AGENT_SYSTEM_PROMPT,
    )

    return agent


print("Person 1 planning agent updated for Groq compatibility.")

Person 1 planning agent updated for Groq compatibility.


In [ ]:
# ============================================================
# PERSON 1 — AGENT FUNDAMENTALS CAPSTONE EVIDENCE
# ============================================================
# This executes the already-defined capstone demo so the saved
# notebook contains first-party evidence of structured output,
# LLM-selected tool calls, tool arguments/results, and final output.

run_capstone_demo()




################################################################################
# SMART EVENT PLANNER — PERSON 1 CAPSTONE DEMO
################################################################################


USER REQUEST

    I want to plan a graduation party for 80 guests
    with a budget of 15,000 SAR.
    The event will be on 20 September 2026.
    I prefer an indoor venue with an elegant theme.
    


STRUCTURED OUTPUT — EVENT REQUEST
{
  "event_type": "graduation",
  "guest_count": 80,
  "budget": 15000.0,
  "date": "20 September 2026",
  "location_pref": "indoor",
  "theme": "elegant"
}


RUNNING REAL LLM AGENT


REAL AGENT TOOL-CALL TRACE

[LLM -> TOOL]
Tool: demo_budget
Arguments:
{'event_type': 'graduation', 'total_budget': 15000}

[LLM -> TOOL]
Tool: demo_venues
Arguments:
{'budget': 15000, 'guest_count': 80, 'indoor': True}

[LLM -> TOOL]
Tool: demo_catering
Arguments:
{'budget': 15000, 'guest_count': 80}

[LLM -> TOOL]
Tool: demo_decoration
Arguments:
{'budget': 15000